# R10 · Source shape assessment: quality and execution cost

This notebook evaluates an unchanged ARM answer **H** using native R08 moments and selected source-cell refinements. The refinement stores exact quadrant quotas and child-local moments. Beta distributions, Gaussian copulas, hierarchical moments, and the small LightGBM controls are classical components. The candidate contribution is the measured combination and its storage/accuracy tradeoff; this notebook establishes no world-first result or bounded heterogeneous merge guarantee.

For a fresh run, provide a completed **R08/R01 run folder** with real histogram/split lineage, a completed R09 run, or explicit histogram/raw/query/grouped-split paths. A Git checkout is `repo_root`, not `run_dir`. A failed R09 experiment is not required. `preparation_dir` can reuse an audited R10 preparation.

The primary package compares 12 KiB and 48 KiB correction caps, keeps the all positive-TV refinements as an upper control, and includes tiny supervised ML controls. A separate full-refinement package freezes before either fresh confirmation. Original test results remain exploratory. Fresh geometry and every assessor score are saved before raw labels; calibration gates remain fixed. Fresh rectangles query the same inspected dataset. Three seeds do not create three independent datasets.


## Notebook-scoped dependencies

Run this cell first. It installs only missing packages, with LightGBM pinned to 4.6.0. Databricks uses notebook-scoped `%pip`; a Python restart may be necessary after installation. If this cell restarts Python, continue with the next cell. An existing incompatible LightGBM installation needs an explicit scoped install and restart rather than an implicit environment upgrade.


In [ ]:
import importlib.metadata
import importlib.util
import subprocess
import sys

missing = [name for name in ("numpy", "pandas", "scipy", "matplotlib")
           if importlib.util.find_spec(name) is None]
if importlib.util.find_spec("lightgbm") is None:
    missing.append("lightgbm==4.6.0")
elif importlib.metadata.version("lightgbm") != "4.6.0":
    raise RuntimeError("Use a notebook-scoped install of lightgbm==4.6.0, then restart Python and rerun this cell.")
if missing:
    if "dbutils" in globals():
        get_ipython().run_line_magic("pip", "install " + " ".join(missing))
        print("Installation finished. Restarting notebook Python; continue with the next cell.")
        dbutils.library.restartPython()
    else:
        subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
print("Required packages are available. LightGBM controls use version 4.6.0.")


## Paths, resume locations, and the explicit execution backend

Set paths with widgets. Outputs default to a runtime-derived user folder outside Git. Direct inputs require `histogram_dir` (`src_<id>.txt` and `arm_<resolution>.txt`), `raw_data_dir` (real TAB/gzip sources), `queries_file` (bounds and offline truth), and an inherited `split_file` with grouped TRAIN/calibration/test memberships. Exact query IDs are preserved.

Resume locations must remain in their original directories: frozen manifests retain absolute provenance paths. Leave a resume field blank to create a new phase. Use the same checkout and package versions for a frozen resume; lock verification fails when scientific source changes.

`execution_backend=compiled` explicitly requires a working g++ and compatible SciPy C API. Compilation failure stops the compiled path. To use the Python indexed implementation, select `indexed` explicitly; no silent fallback changes the benchmark. Both retain Python request orchestration and genuine resident raw-array scan fallback.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from uuid import uuid4
import importlib.metadata
import json
import os
import shutil
import sys


def detect_repo_root():
    candidates = [Path.cwd(), *Path.cwd().parents]
    try:
        notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
        here = Path("/Workspace") / notebook_path.lstrip("/")
        candidates += [here.parent, *here.parents]
    except Exception:
        pass
    for candidate in candidates:
        if (candidate / "ml" / "shape_inputs.py").is_file():
            return candidate.resolve()
    return Path.cwd().resolve()


def default_output_root(repo):
    try:
        user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().userName().get()
        return str(Path("/Workspace/Users") / user / "synopsislake-results" / "R10")
    except Exception:
        return str(repo.parent / "synopsislake-results" / "R10")


detected_repo = detect_repo_root()
defaults = {
    "repo_root": str(detected_repo), "run_dir": "", "histogram_dir": "",
    "raw_data_dir": str(detected_repo / "range_query"), "queries_file": "",
    "query_metadata_file": "", "split_file": "", "base_resolution": "130",
    "output_root": default_output_root(detected_repo), "preparation_dir": "",
    "compact_development_dir": "", "full_development_dir": "",
    "compact_confirmation_dir": "", "full_confirmation_dir": "",
    "compact_execution_dir": "", "full_execution_dir": "",
    "run_fresh_confirmation": "true", "run_execution": "true", "execution_backend": "compiled",
}
try:
    for key, value in defaults.items():
        try:
            dbutils.widgets.get(key)
        except Exception:
            dbutils.widgets.text(key, value)
    settings = {key: dbutils.widgets.get(key).strip() for key in defaults}
except NameError:
    settings = dict(defaults)
repo_root = Path(settings["repo_root"]).expanduser().resolve()
if not (repo_root / "ml" / "shape_inputs.py").is_file():
    raise FileNotFoundError("Set repo_root to the checkout containing ml/shape_inputs.py.")
output_root = Path(settings["output_root"]).expanduser().resolve()
if output_root == repo_root or repo_root in output_root.parents:
    raise ValueError("output_root must be outside the Git checkout.")
output_root.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(output_root / "matplotlib-cache"))
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
from ml import shape_inputs
from ml import local_shape_workflow as workflow
from ml import shape_execution
from ml import sidecar_workflow as prior
import numpy as np
import pandas as pd


def boolean_setting(name):
    value = settings[name].lower()
    if value not in ("true", "false"):
        raise ValueError(name + " must be true or false")
    return value == "true"

backend = settings["execution_backend"]
if backend not in ("compiled", "indexed"):
    raise ValueError("execution_backend must be compiled or indexed")
if boolean_setting("run_execution") and backend == "compiled" and shutil.which("g++") is None:
    raise RuntimeError("Compiled execution requires g++. Install it in the runtime or explicitly select execution_backend=indexed.")
bundle_dir = output_root / ("R10-review-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "-" + uuid4().hex[:8])
bundle_dir.mkdir(exist_ok=False)
print("Repository:", repo_root)
print("Outputs:", output_root)
print("This review bundle:", bundle_dir)


## Audit real inputs once

`prepare_inputs` accepts direct real exports or inherited R08/R09 paths. It verifies exact source-cell masses, stores source-only moments/quadrant counts, and checks a deterministic sample of stored truth against inclusive raw scans. Raw coordinates are parsed into private caches; later exact labeling and fallback use those same verified coordinates. The raw cache is not exported.

A grouped outer split is mandatory for this study. No synthetic replacement dataset or newly invented split is substituted for missing inputs.


In [ ]:
input_config = {
    "repo_root": str(repo_root), "output_root": str(output_root),
    "run_dir": settings["run_dir"], "histogram_dir": settings["histogram_dir"],
    "raw_data_dir": settings["raw_data_dir"], "queries_file": settings["queries_file"],
    "query_metadata_file": settings["query_metadata_file"], "split_file": settings["split_file"],
    "split_kind": "cell", "base_resolution": int(settings["base_resolution"]),
    "preparation_dir": settings["preparation_dir"], "seeds": [0, 1, 2],
    "min_exact": 10, "chunk_size": 250000,
}
preparation_dir = shape_inputs.prepare_inputs(input_config)
preparation_dir, preparation_state = workflow._load_preparation(preparation_dir)
print("Audited preparation:", preparation_dir)
print("Real rows:", preparation_state["raw_rows"])
print("Native auxiliary bytes:", preparation_state["native_bytes"])
source_audit = pd.read_csv(preparation_dir / "source_audit.csv")
display(source_audit) if "display" in globals() else print(source_audit.to_string(index=False))


## Freeze both development packages before fresh geometry

Every seed uses all of its inherited outer TRAIN. The deterministic grouped 80/20 inner split selects the allowed correction cap by macro-seed proxy MAE; models select their fixed small tree sizes on the same declared inner roles and save validation predictions before final TRAIN refitting. Calibration truth chooses gates only. Existing test caps are 512 queries per seed and remain exploratory.

The primary curve measures extra caps 12,288 / 49,152 / 1,000,000,000 bytes. Only the first two can be promoted under the declared compact constraint; the large ceiling materializes an actual packet for every positive-TV cell and is reported at its actual bytes. Native R08 and same-selection empirical-uniform controls remain visible. Primary tiny-ML controls charge model text and schema. Every finer-grid comparator matches actual native + correction + model/schema auxiliary bytes.

The secondary positive-TV full package is a prespecified upper reference with no ML. It is frozen now, before either confirmation. Original H is common and unchanged throughout. If native R08 wins or the declared validation criterion fails, the corresponding fresh phase is skipped; this notebook never forces promotion.


In [ ]:
common_development = dict(
    family="hierarchical", preparation_dir=str(preparation_dir), output_root=str(output_root),
    seeds=[0, 1, 2], train_queries=0, validation_queries=0,
    calibration_queries=512, test_queries=512, epsilon=0.25, delta=0.05, min_exact=10,
    evaluate_old_test=True, timing_queries=0, fresh_query_count=0,
)
compact_config = dict(common_development, extra_budgets=[12288, 49152, 1000000000],
                      selection_budgets=[12288, 49152], include_ml=True)
full_config = dict(common_development, extra_budgets=[1000000000],
                   selection_budgets=[1000000000], include_ml=False)


def development_phase(resume_value, config):
    if not resume_value:
        return workflow.develop_experiment(config)
    path = Path(resume_value).expanduser().resolve()
    workflow.verify_protocol_lock(path)
    state = json.loads((path / "run.json").read_text())
    if state.get("status") != "complete":
        raise ValueError("Resume requires a complete development run: " + str(path))
    if Path(state["preparation_dir"]).resolve() != preparation_dir:
        raise ValueError("Resume development uses a different preparation directory.")
    for key in ("family", "seeds", "train_queries", "validation_queries", "calibration_queries",
                "test_queries", "min_exact", "evaluate_old_test", "extra_budgets", "selection_budgets", "include_ml", "epsilon", "delta"):
        if state["config"][key] != config[key]:
            raise ValueError("Resume development setting differs: " + key)
    return path

compact_development = development_phase(settings["compact_development_dir"], compact_config)
full_development = development_phase(settings["full_development_dir"], full_config)
for path in (compact_development, full_development):
    workflow.verify_protocol_lock(path)
paired_freeze = {
    "status": "both_packages_frozen_before_confirmation", "fresh_query_seed": 2026111801,
    "fresh_query_count": 1024, "quality_timing_queries": 0,
    "packages": {name: {"path": str(path), "protocol_sha256": prior._sha(path / "protocol.json")}
                 for name, path in (("compact", compact_development), ("full", full_development))},
}
prior._json(bundle_dir / "paired_freeze.json", paired_freeze)
for name, path in (("compact", compact_development), ("full", full_development)):
    print(name, path)
    print((path / "selection.json").read_text())
    table = pd.read_csv(path / "validation_macro.csv")
    display(table) if "display" in globals() else print(table.to_string(index=False))


## Fresh quality confirmation, without timing

The confirmation uses 1,024 rectangles and seed 2026111801. Width/height donors come only from outer TRAIN. Both packages use identical donors and geometry; the notebook verifies geometry identity. All method scores and the score lock are persisted before exact indexed raw labels. Gates are copied from original calibration and never fitted to fresh truth.

The same 1,024 geometries are reused across seeds and the two packages. They are not 3,072 or 6,144 independent test queries. Repeating this seed reproduces the same cohort. Labels come from original points, with independent raw-scan audits.


In [ ]:
def confirmation_phase(name, development, resume_value):
    for frozen in (compact_development, full_development):
        workflow.verify_protocol_lock(frozen)
    selection = json.loads((development / "selection.json").read_text())
    if not selection["promising"]:
        print(name + ": fresh confirmation skipped because the declared validation criterion failed.")
        return None
    if not boolean_setting("run_fresh_confirmation"):
        print(name + ": fresh confirmation disabled by widget.")
        return None
    if resume_value:
        path = Path(resume_value).expanduser().resolve()
        protocol = workflow.verify_confirmation_lock(path)
        state = json.loads((path / "run.json").read_text())
        if state.get("status") != "complete" or Path(state["development_dir"]).resolve() != development:
            raise ValueError("Resume confirmation must be complete and belong to the selected development.")
        if (protocol["config"]["fresh_query_count"] != 1024 or
            protocol["config"]["fresh_query_seed"] != 2026111801 or
            protocol["config"]["timing_queries"] not in (0, 32) or state.get("forced_exploratory")):
            raise ValueError("Resume confirmation differs from the frozen notebook protocol.")
        print(name + ": resumed quality; original timing limit:", protocol["config"]["timing_queries"])
        return path
    return workflow.confirm_experiment(dict(
        development_dir=str(development), fresh_query_count=1024, fresh_query_seed=2026111801,
        timing_queries=0, force_confirmation=False,
    ))

compact_confirmation = confirmation_phase("compact", compact_development, settings["compact_confirmation_dir"])
full_confirmation = confirmation_phase("full", full_development, settings["full_confirmation_dir"])
if compact_confirmation is not None and full_confirmation is not None:
    first = pd.read_csv(compact_confirmation / "fresh_queries_unlabeled.csv", float_precision="round_trip")
    second = pd.read_csv(full_confirmation / "fresh_queries_unlabeled.csv", float_precision="round_trip")
    pd.testing.assert_frame_equal(first, second, check_exact=True)
    print("Both packages used the same 1,024 fresh geometries.")
confirmations = {name: path for name, path in (("compact", compact_confirmation), ("full", full_confirmation))
                 if path is not None}


## Separate executable setup from live request measurements

Quality confirmation has finished before this phase. The replay chooses 32 query IDs per seed by the locked geometry-ID rule and runs three repetitions in both `resident` and `per_query` modes. Every live score must match its score-before-labels record within the declared tolerance, and gate decisions must match exactly. Every rejection and all-exact baseline perform genuine closed-bound vector scans of resident raw coordinates.

Compiled analytic references use shared C++ code and SciPy's C API; tiny-ML feature extraction retains the indexed Python path and original model inference. The shared executable, compiler identity, and compilation/library setup are separate from data payload bytes. Setup is not request latency.

Resident mode retains loaded H and assessor artifacts. Per-query mode reloads local H and assessor payloads inside the request timer. Raw arrays are resident in both modes and for the exact baseline. First requests and artifact initialization are separate outputs. These are local file reloads with uncontrolled OS caches, not an OS-cold benchmark or a Java/Spark integration. Read speed and accepted-answer violation together.


In [ ]:
execution_setup = {"backend": backend, "execution_enabled": boolean_setting("run_execution")}
if boolean_setting("run_execution") and confirmations:
    execution_setup["kernel_setup"] = shape_execution._backend_setup(backend)
prior._json(bundle_dir / "executable_setup.json", execution_setup)
print(json.dumps(execution_setup, indent=2, default=str))


def execution_phase(confirmation, resume_value):
    workflow.verify_confirmation_lock(confirmation)
    if resume_value:
        path = Path(resume_value).expanduser().resolve()
        state = json.loads((path / "run.json").read_text())
        if (state.get("status") != "complete" or state.get("backend") != backend or
            Path(state["source_confirmation"]).resolve() != confirmation or
            state["timing_queries"] != 32 or state["repeats"] != 3 or
            not state.get("all_live_outputs_and_decisions_match")):
            raise ValueError("Resume execution differs from the selected frozen replay.")
        if state["source_confirmation_protocol_sha256"] != prior._sha(confirmation / "protocol.json"):
            raise ValueError("Resume execution refers to a different confirmation protocol.")
        if state["execution_implementation_sha256"] != shape_execution._source_hashes(backend):
            raise ValueError("Resume execution uses a different implementation.")
        for filename, key in (
            ("run.json", "source_confirmation_manifest_sha256"),
            ("fresh_score_lock.json", "source_score_lock_sha256"),
            ("fresh_queries_unlabeled.csv", "source_geometry_sha256"),
            ("fresh_scores_before_labels.csv", "source_scores_sha256"),
            ("frozen_gate_cutoffs.csv", "source_gate_sha256"),
            ("fresh_queries_labeled_offline.csv", "source_labels_sha256"),
        ):
            if prior._sha(confirmation / filename) != state[key]:
                raise ValueError("Resume provenance changed: " + filename)
        measurements = []
        for file in sorted((path / "e2e").glob("seed_*/per_query.csv")):
            seed = int(file.parent.name.split("_")[-1])
            for filename in ("per_query.csv", "first_query.csv"):
                frame = pd.read_csv(file.parent / filename, float_precision="round_trip")
                measurements.append(frame.assign(seed=seed))
        if not measurements:
            raise ValueError("Resume execution contains no live request records.")
        parity = shape_execution.verify_live_parity(
            pd.concat(measurements, ignore_index=True),
            pd.read_csv(confirmation / "fresh_scores_before_labels.csv", float_precision="round_trip"),
            pd.read_csv(confirmation / "frozen_gate_cutoffs.csv", float_precision="round_trip"),
        )
        if not parity.all_matches.all():
            raise ValueError("Resume live outputs differ from frozen scores or gates.")
        return path
    return shape_execution.run_indexed_confirmation(
        confirmation, output_root, timing_queries=32, repeats=3, backend=backend,
    )

executions = {}
if boolean_setting("run_execution"):
    for name, confirmation in confirmations.items():
        executions[name] = execution_phase(confirmation, settings[name + "_execution_dir"])
        print(name + " live execution:", executions[name])
else:
    print("Execution disabled; quality results do not establish E2E speed.")


## English tables and figures

Error-prediction MAE is in percentage points and measures the assessor score against the true relative error of the unchanged H. AUC measures bad-answer ranking. The numerical MAE plot uses the declared truth≥10 cohort because a positive answer at zero truth has infinite relative error. The all-query, small-count, and zero-count rows remain visible below. Zero accepted answers have undefined accepted violation.

The 10/25/50/100% fixed-coverage tables retain complete score ties and are ranking diagnostics. Empirical calibration targets ε=25% and δ=5%; neither calibration nor these tables provides a five-percent future-risk guarantee. Same-dataset fresh results do not establish transfer to another population. Models and all unsuccessful controls remain in the tables.

Latency aggregation below pools actual request observations. It reports the ratio of pooled mean exact time to pooled mean method time, and computes p95 from those observations. It does not average per-seed speedup ratios or per-seed p95 values. Repeats and seed appearances are not independent query samples. The numerical cohorts use truth only after execution for reporting.


In [ ]:
quality_tables, cost_tables, latency_tables, decision_rows = [], [], [], []
for package, development in (("compact", compact_development), ("full", full_development)):
    print(package + " allocation and stored-byte audit")
    for filename in ("allocation_summary.csv", "payload_costs.csv", "validation_metrics.csv"):
        table = pd.read_csv(development / filename)
        display(table) if "display" in globals() else print(table.to_string(index=False))
    cost_tables.append(pd.read_csv(development / "payload_costs.csv").assign(package=package))

for package, confirmation in confirmations.items():
    for filename in ("metrics.csv", "fixed_coverage.csv", "fresh_raw_label_audit.csv"):
        table = pd.read_csv(confirmation / filename)
        print(package, filename)
        display(table) if "display" in globals() else print(table.to_string(index=False))
    quality_tables.append(pd.read_csv(confirmation / "metrics.csv").assign(package=package))
    saved = pd.read_csv(confirmation / "fresh_scores_before_labels.csv", float_precision="round_trip")
    labels = pd.read_csv(confirmation / "fresh_queries_labeled_offline.csv", float_precision="round_trip")
    gates = pd.read_csv(confirmation / "frozen_gate_cutoffs.csv", float_precision="round_trip")
    geometry = pd.read_csv(confirmation / "fresh_queries_unlabeled.csv", float_precision="round_trip")
    shape_execution._geometry_labels(geometry, labels)
    decisions = saved.merge(labels[["qId", "truth"]], on="qId", validate="many_to_one")
    decisions = decisions.merge(gates[["seed", "method", "cutoff"]],
                                on=["seed", "method"], validate="many_to_one")
    decisions["actual_error"] = prior._actual_error(decisions.H, decisions.truth)
    decisions["accepted"] = np.isfinite(decisions.score) & (decisions.score <= decisions.cutoff)
    decisions["bad"] = decisions.actual_error > 0.25
    for (seed, method), g in decisions.groupby(["seed", "method"]):
        for cohort, mask in (
            ("all", np.ones(len(g), dtype=bool)), ("truth_ge_10", g.truth >= 10),
            ("positive_truth_lt_10", (g.truth > 0) & (g.truth < 10)), ("zero_truth", g.truth == 0),
        ):
            x = g[mask]
            accepted = int(x.accepted.sum())
            bad = int((x.accepted & x.bad).sum())
            decision_rows.append(dict(package=package, seed=int(seed), method=method, cohort=cohort,
                query_count=len(x), accepted_count=accepted, accepted_bad_count=bad,
                accepted_good_count=accepted-bad, acceptance=accepted/len(x) if len(x) else np.nan,
                accepted_violation=bad/accepted if accepted else np.nan))
if decision_rows:
    decision_counts = pd.DataFrame(decision_rows)
    decision_counts.to_csv(bundle_dir / "fresh_decision_counts.csv", index=False)
    print("Reconstructed decisions from locked pre-label scores and original calibration gates:")
    display(decision_counts) if "display" in globals() else print(decision_counts.to_string(index=False))

for package, execution in executions.items():
    measurements = []
    for file in sorted((execution / "e2e").glob("seed_*/per_query.csv")):
        frame = pd.read_csv(file, float_precision="round_trip")
        measurements.append(frame.assign(seed=int(file.parent.name.split("_")[-1])))
    live = pd.concat(measurements, ignore_index=True)
    for (cache_mode, method), g in live.groupby(["cache_mode", "method"]):
        exact = live[(live.cache_mode == cache_mode) & (live.method == "always_exact")]
        cases = g.drop_duplicates(["seed", "qId"])
        accepted = int(cases.accepted.sum())
        bad = int(cases.accepted_violation.sum())
        latency_tables.append(dict(package=package, cache_mode=cache_mode, method=method,
            observations=len(g), unique_query_ids=g.qId.nunique(), seed_query_cases=len(cases),
            repeats=g["repeat"].nunique(), accepted_count=accepted, accepted_bad_count=bad,
            timing_subset_violation=bad/accepted if accepted else np.nan,
            total_mean_ms=g.total_ns.mean()/1e6, total_p95_ms=g.total_ns.quantile(0.95)/1e6,
            exact_mean_ms=exact.total_ns.mean()/1e6,
            speedup_ratio_of_pooled_means=exact.total_ns.mean()/g.total_ns.mean()))
    print("Separate per-seed summaries, first requests, and initialization:", execution / "e2e")

import matplotlib.pyplot as plt
if quality_tables:
    quality = pd.concat(quality_tables, ignore_index=True)
    quality.to_csv(bundle_dir / "fresh_quality_all_cohorts.csv", index=False)
    eligible = quality[quality.cohort == "truth_ge_min_exact"]
    summary = eligible.groupby(["package", "method"], sort=False).agg(
        mae_pp=("mae_pp", "mean"), auc_bad=("auc_bad", "mean"),
        coverage=("coverage", "mean"), accepted_violation=("violation", "mean"),
        minimum_queries=("n_queries", "min"))
    summary.to_csv(bundle_dir / "fresh_quality_macro.csv")
    fig, axes = plt.subplots(1, 2, figsize=(14, 8), constrained_layout=True)
    summary.mae_pp.plot.barh(ax=axes[0], color="#e08b39")
    summary.auc_bad.plot.barh(ax=axes[1], color="#337ab7")
    axes[0].set_xlabel("Error-prediction MAE, percentage points; truth >= 10")
    axes[1].set_xlabel("Bad-answer AUC; truth >= 10")
    fig.savefig(bundle_dir / "fresh_quality.png", dpi=160)
    plt.show()
    plt.close(fig)
    fig, axes = plt.subplots(1, len(confirmations), figsize=(7*len(confirmations), 4),
                             squeeze=False, constrained_layout=True)
    for ax, (package, confirmation) in zip(axes[0], confirmations.items()):
        curves = pd.read_csv(confirmation / "fixed_coverage.csv")
        for method in ("R08", "hybrid", "fine_for_hybrid"):
            x = curves[(curves.method == method) & (curves.cohort == "truth_ge_min_exact")]
            line = x.groupby("nominal_coverage").violation.mean()
            ax.plot(line.index*100, line.values*100, "o-", label=method)
        ax.set(title=package + ": ranking diagnostic", xlabel="Nominal fraction accepted (%)",
               ylabel="Bad fraction among accepted (%)")
        ax.legend()
    fig.savefig(bundle_dir / "equal_coverage_risk.png", dpi=160)
    plt.show()
    plt.close(fig)
if latency_tables:
    latency = pd.DataFrame(latency_tables)
    latency.to_csv(bundle_dir / "live_e2e_pooled.csv", index=False)
    print("Pooled live-request latency; interpret timing-subset risk with full-cohort quality:")
    display(latency) if "display" in globals() else print(latency.to_string(index=False))
    means = latency.set_index(["package", "method", "cache_mode"]).total_mean_ms
    fig, ax = plt.subplots(figsize=(12, 10), constrained_layout=True)
    means.plot.barh(ax=ax, color="#4c9b61")
    ax.set_xlabel("Pooled mean live E2E milliseconds, including raw-scan fallback")
    fig.savefig(bundle_dir / "live_e2e_cost.png", dpi=160)
    plt.show()
    plt.close(fig)
pd.concat(cost_tables, ignore_index=True).to_csv(bundle_dir / "payload_costs.csv", index=False)


## Export a review packet outside Git

The curated archive contains both development packages, available confirmations and live replays, their English analyses/tables, source/packet/model artifacts, source-only construction audits, and copies of the implementation used. It excludes raw-coordinate caches, feature caches, existing ZIPs, and unrelated nested confirmation runs. No raw point file is exported.

The bundle records paths and hashes for provenance; resume uses the original run directories. The archive is review evidence, not a relocated runtime installation. Read accepted coverage/violation, numerical score quality, stored bytes, setup, and request cost together. Training is confined to explicit ML controls; the analytic assessor has no supervised fitting and leaves H unchanged.


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

phases = {"compact_development": compact_development, "full_development": full_development}
phases.update({name + "_confirmation": path for name, path in confirmations.items()})
phases.update({name + "_execution": path for name, path in executions.items()})
manifest = {
    "created_utc": datetime.now(timezone.utc).isoformat(), "backend": backend,
    "real_rows": preparation_state["raw_rows"], "fresh_query_seed": 2026111801,
    "fresh_geometry_count": 1024 if confirmations else 0, "independent_dataset": False,
    "training": "only the explicitly reported tiny-ML controls",
    "paths": {name: str(path) for name, path in phases.items()},
    "protocol_sha256": {name: prior._sha(path / "protocol.json") for name, path in phases.items()
                        if (path / "protocol.json").is_file()},
    "package_versions": {name: importlib.metadata.version(name)
                         for name in ("numpy", "pandas", "scipy", "matplotlib", "lightgbm")},
}
prior._json(bundle_dir / "manifest.json", manifest)
(bundle_dir / "analysis_en.md").write_text(
    "# R10 review packet\n\nBoth development packages were frozen before fresh confirmation. "
    "Candidate cap selection uses grouped inner validation inside original TRAIN. Fresh scores precede "
    "indexed raw labels; gates remain fixed. The primary compact package includes negative ML/uniform "
    "controls, while full refinement is a prespecified upper control. See the per-seed CSVs; repeated "
    "seeds and packages reuse geometry on the same inspected dataset. These classical components do not "
    "establish novelty, a bounded merge algorithm, a five-percent guarantee, or a cloud E2E speed guarantee. Any local speed claim must cite the measured backend, cache mode, and quality rows. "
    "Compilation/setup, local artifact reloads, and resident request measurements remain separate.\n"
)
archive_path = bundle_dir / "R10_source_shape_results.zip"
with ZipFile(archive_path, "w", ZIP_DEFLATED) as zipped:
    for name, root in phases.items():
        for item in sorted(root.rglob("*")):
            relative = item.relative_to(root)
            if (not item.is_file() or item.suffix == ".zip" or
                set(relative.parts) & {"raw_cache", "feature_cache", "confirmation", "development"}):
                continue
            zipped.write(item, str(Path(name) / relative))
    for filename in ("run.json", "source_audit.csv", "raw_truth_parity.csv", "input_manifest.json"):
        item = preparation_dir / filename
        if item.is_file():
            zipped.write(item, str(Path("source_preparation") / filename))
    implementation_names = (
        "shape_inputs.py", "local_shape_workflow.py", "local_shape.py", "hierarchical_shape.py",
        "shape_error_model.py", "shape_runtime.py", "shape_execution.py", "shape_kernel.py", "shape_kernel.cc",
        "sidecar_workflow.py", "sidecar_e2e.py", "bounded_sidecar.py", "_distribution_math.py", "features.py",
    )
    for filename in implementation_names:
        item = repo_root / "ml" / filename
        if item.is_file():
            zipped.write(item, str(Path("implementation/ml") / filename))
    for item in sorted(bundle_dir.iterdir()):
        if item.is_file() and item.suffix != ".zip":
            zipped.write(item, item.name)
print("Share:", archive_path)
print("Resume paths and frozen manifests:", bundle_dir / "manifest.json")
